# 🤖 Giorno 3 - Lab 2: Costruire l'MVP — Agent + Tools con LangChain

Benvenuti al secondo laboratorio del **Giorno 3**!

Nel Lab 1 abbiamo esplorato i blocchi fondamentali di LangChain. Adesso li usiamo per costruire l'MVP completo.

**Vantaggi di LangChain rispetto al loop manuale del Lab 2 originale:**
- `@tool` — genera lo schema JSON **automaticamente** da docstring e type hints
- `AgentExecutor` — gestisce il loop ReAct internamente, nessun `for` manuale
- `ChatOllama` — integrazione nativa con Ollama, nessuna chiamata `requests.post`
- `verbose=True` — tracciamento built-in del ragionamento dell'agente

---
### 🎯 Obiettivo di Lab
Al termine il flow deve rispondere correttamente ad almeno **3 richieste con esiti diversi**:
1. ✅ **Approvazione automatica** — richiesta in policy, budget sufficiente
2. ⚠️  **Escalation al manager** — fuori fascia o budget insufficiente
3. ❌ **Rifiuto** — prodotto non a catalogo

---
### 🗺️ Pattern: Agent + Tools (LangChain)
```
input (str)
  │
  ▼
AgentExecutor
  ├─ create_tool_calling_agent(ChatOllama, tools, prompt)
  │    ├─ @tool  get_employee_budget
  │    └─ @tool  search_product_catalog
  └─ loop: reason → tool_call → observe → repeat → final answer
  │
  ▼
risultato["output"]
```

## 1. Setup

In [1]:
import json
from pathlib import Path

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
try:
    from langchain.agents import AgentExecutor, create_tool_calling_agent
except ImportError:
    from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

DATA_DIR = Path("data")

# LLM — ChatOllama sostituisce interamente la chiamata requests.post
llm = ChatOllama(model="llama3.2", temperature=0)
print("✅ LangChain + ChatOllama pronti.")
print(f"   Modello: {llm.model}")

✅ LangChain + ChatOllama pronti.
   Modello: llama3.2


## 2. Definizione dei Tool con `@tool`

Il decorator `@tool` di LangChain:
- Trasforma qualsiasi funzione Python in un tool compatibile con `AgentExecutor`
- **Genera automaticamente lo schema JSON** dalla docstring e dai type hints
- Espone `.name`, `.description` e `.args_schema` usati internamente dall'agente

> ⚠️ **La docstring è il tuo schema**: più è chiara e specifica, più il modello saprà *quando* chiamare il tool.

In [2]:
# ── TOOL 1: get_employee_budget ──────────────────────────────────────────
@tool
def get_employee_budget(employee_id: str) -> dict:
    """Recupera budget residuo, ruolo, dipartimento e progetto attivo di un dipendente.
    Chiamare SEMPRE prima di valutare qualsiasi richiesta per conoscere budget disponibile
    e fascia di spesa autorizzata. Argomento: employee_id (badge ID, es. 'E4521')."""
    employees = json.loads((DATA_DIR / "employees.json").read_text())
    if employee_id not in employees:
        return {"status": "NOT_FOUND", "message": f"Dipendente '{employee_id}' non trovato."}
    emp = employees[employee_id]
    return {
        "status": "OK",
        "employee_id": employee_id,
        "name": emp["name"],
        "department": emp["department"],
        "role": emp["role"],
        "budget_residuo_eur": emp["budget_residuo"],
        "active_project": emp.get("active_project"),
        "manager_email": emp["manager_email"]
    }


# ── TOOL 2: search_product_catalog ───────────────────────────────────────
@tool
def search_product_catalog(query: str) -> dict:
    """Cerca prodotti nel catalogo hardware aziendale per nome o categoria.
    Usare SEMPRE per verificare se il prodotto richiesto esiste a catalogo, il suo prezzo ufficiale e la disponibilità.
    Categorie disponibili: laptop, monitor, accessory.
    Esempi di query: 'ThinkPad', 'laptop', 'monitor', 'Dell'."""
    products = json.loads((DATA_DIR / "products.json").read_text())
    q = query.lower().strip()
    results = {}
    for pid, p in products.items():
        name_l = p["name"].lower()
        cat_l = p["category"].lower()
        roles_l = [r.lower() for r in p["target_roles"]]
        if q in name_l or q in cat_l or any(q == r for r in roles_l):
            results[pid] = p
    if not results:
        terms = [t for t in q.split() if t not in ["il", "la", "un", "una", "del", "per", "di"]]
        for pid, p in products.items():
            name_l = p["name"].lower()
            if sum(1 for t in terms if t in name_l or t in p["category"].lower()) >= 2:
                results[pid] = p
    if not results:
        return {"status": "NOT_FOUND", "message": f"Nessun prodotto trovato per '{query}'. Prodotto NON presente a catalogo."}
    return {"status": "OK", "risultati": results, "totale": len(results)}


# Ispeziona lo schema generato automaticamente
print("📋 Schema generato da @tool (automatico, no JSON manuale):")
for t in [get_employee_budget, search_product_catalog]:
    print(f"\n  Tool: {t.name}")
    print(f"  Descr: {t.description[:80]}...")
    print(f"  Args:  {t.args}")


📋 Schema generato da @tool (automatico, no JSON manuale):

  Tool: get_employee_budget
  Descr: Recupera budget residuo, ruolo, dipartimento e progetto attivo di un dipendente....
  Args:  {'employee_id': {'title': 'Employee Id', 'type': 'string'}}

  Tool: search_product_catalog
  Descr: Cerca prodotti nel catalogo hardware aziendale per nome o categoria.
    Usare S...
  Args:  {'query': {'title': 'Query', 'type': 'string'}}


## 3. Il Prompt Template dell'Agente

Il `ChatPromptTemplate` definisce:
- Il **system message** con ruolo + policy (contesto)
- Il placeholder `{input}` per la richiesta dell'utente
- Il placeholder `agent_scratchpad` — **obbligatorio**: è qui che l'agente scrive i suoi ragionamenti intermedi (tool calls e risultati)

La policy aziendale viene caricata come testo e iniettata nel system prompt.

In [3]:
policy_text = (DATA_DIR / "policy_hardware.md").read_text()

SYSTEM_PROMPT = f"""Sei l'Assistente Aziendale per gli Acquisti Hardware.

Sequenza operativa obbligatoria prima di rispondere:
1. Chiama 'get_employee_budget' per ottenere budget residuo, ruolo e progetto attivo del dipendente.
2. Chiama SEMPRE 'search_product_catalog' per verificare se il prodotto richiesto è presente a catalogo e il suo prezzo esatto.
3. Valuta le informazioni ottenute secondo queste regole:
   - RIFIUTO: se 'search_product_catalog' restituisce NOT_FOUND (prodotto NON a catalogo). Non inventare prezzi, modelli o approvazioni per prodotti non presenti nel catalogo aziendale.
   - APPROVAZIONE AUTOMATICA: se il prodotto è a catalogo, rientra nella fascia di spesa per il ruolo e il costo è interamente coperto dal budget residuo.
   - ESCALATION AL MANAGER: se il prodotto è a catalogo ma supera la fascia standard del ruolo (pur con progetto attivo) oppure supera il budget residuo.
4. Rispondi SEMPRE in italiano indicando chiaramente la decisione (APPROVAZIONE AUTOMATICA, ESCALATION AL MANAGER o RIFIUTO) e la motivazione.

=== POLICY AZIENDALE ===
{policy_text}
=======================
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

print("✅ Prompt template creato.")
print("   Variabili:", prompt.input_variables)


✅ Prompt template creato.
   Variabili: ['agent_scratchpad', 'input']


## 4. Assemblaggio dell'Agente MVP

Tre righe di LangChain sostituiscono l'intero loop ReAct del Lab 2 originale:

```python
agent    = create_tool_calling_agent(llm, tools, prompt)  # il ragionatore
executor = AgentExecutor(agent=agent, tools=tools, ...)    # il loop
result   = executor.invoke({"input": richiesta})           # l'esecuzione
```

In [4]:
tools = [get_employee_budget, search_product_catalog]

# Agente: collega LLM, tool e prompt
agent = create_tool_calling_agent(llm, tools, prompt)

# Executor: gestisce il loop ReAct
# verbose=True → mostra ogni step del ragionamento (tool chiamato, argomenti, risultato)
executor = AgentExecutor(
    agent   = agent,
    tools   = tools,
    verbose = True,
    max_iterations = 8,
    handle_parsing_errors = True,
)

print("✅ AgentExecutor assemblato.")
print(f"   Tool disponibili: {[t.name for t in tools]}")

✅ AgentExecutor assemblato.
   Tool disponibili: ['get_employee_budget', 'search_product_catalog']


## 5. Test del Flow MVP — 3 Scenari di Successo

---
### 📋 Scenario 1: Approvazione Automatica
**Mario Rossi (E4521)** — Engineering, budget 1800€, ThinkPad P16s a 1650€.
Atteso: ✅ APPROVAZIONE AUTOMATICA

In [5]:
print("=" * 60)
print("  SCENARIO 1 — Approvazione automatica")
print("=" * 60)

r1 = executor.invoke({"input": (
    "Ciao! Sono Mario Rossi, badge E4521. "
    "Lavoro su Project Delta e ho bisogno di un laptop performante per sviluppo. "
    "Posso acquistare il ThinkPad P16s? Rientra nella policy?"
)})

print("\n📤 RISPOSTA FINALE:", r1["output"])

  SCENARIO 1 — Approvazione automatica


> Entering new AgentExecutor chain...

Invoking: `get_employee_budget` with `{'employee_id': 'E4521'}`


{'status': 'OK', 'employee_id': 'E4521', 'name': 'Mario Rossi', 'department': 'Engineering', 'role': 'Senior Developer', 'budget_residuo_eur': 1800.0, 'active_project': 'Project Delta', 'manager_email': 'chiara.bianchi@azienda.it'}
Invoking: `search_product_catalog` with `{'query': 'ThinkPad P16s'}`


{'status': 'OK', 'risultati': {'LAP-DEV-01': {'name': 'ThinkPad P16s Gen 2 / MacBook Pro M3', 'category': 'laptop', 'target_roles': ['Engineering', 'Data Science'], 'price_eur': 1650.0, 'specs': '32GB RAM, 1TB SSD, 16-inch display, GPU integrata/dedicata', 'in_stock': True}}, 'totale': 1}Ciao Mario Rossi,

La tua richiesta di acquisto di un laptop ThinkPad P16s per lo sviluppo è stata approvata con Approvazione Automatica.

Il ThinkPad P16s rientra nella fascia di spesa standard per il tuo ruolo di Senior Developer e il costo del prodotto (€1650

---
### 📋 Scenario 2: Escalation al Manager
**Giulia Verdi (E1082)** — Marketing, budget 450€, monitor 4K Dell a 520€.
Atteso: ⚠️ ESCALATION AL MANAGER

In [6]:
print("=" * 60)
print("  SCENARIO 2 — Escalation al manager")
print("=" * 60)

r2 = executor.invoke({"input": (
    "Salve, sono Giulia Verdi (badge E1082) del team Marketing. "
    "Per la Brand Campaign Q2 ho bisogno del monitor 4K Dell UltraSharp. "
    "Posso procedere all'acquisto?"
)})

print("\n📤 RISPOSTA FINALE:", r2["output"])

  SCENARIO 2 — Escalation al manager


> Entering new AgentExecutor chain...

Invoking: `search_product_catalog` with `{'query': 'monitor 4K Dell UltraSharp'}`


{'status': 'OK', 'risultati': {'MON-4K-01': {'name': 'Dell UltraSharp 27 4K (U2723QE)', 'category': 'monitor', 'target_roles': ['All'], 'price_eur': 520.0, 'specs': '27-inch 4K IPS, USB-C Hub 90W power delivery', 'in_stock': True}}, 'totale': 1}
Invoking: `get_employee_budget` with `{'employee_id': 'E1082'}`


{'status': 'OK', 'employee_id': 'E1082', 'name': 'Giulia Verdi', 'department': 'Marketing', 'role': 'Content Specialist', 'budget_residuo_eur': 450.0, 'active_project': 'Brand Campaign Q2', 'manager_email': 'luca.neri@azienda.it'}Ciao Giulia,

La tua richiesta di acquisto del monitor 4K Dell UltraSharp è stata valutata in base alla policy aziendale per gli acquisti hardware e dotazioni IT.

Il prodotto è presente nel catalogo aziendale e il prezzo è di 520€. Tuttavia, il costo supera la fascia standard assegnata al tuo r

---
### 📋 Scenario 3: Rifiuto — Prodotto Non a Catalogo
Atteso: ❌ RIFIUTO

In [7]:
print("=" * 60)
print("  SCENARIO 3 — Prodotto non a catalogo")
print("=" * 60)

r3 = executor.invoke({"input": (
    "Ciao, sono Alessandro Corti, badge E9900. "
    "Ho bisogno di un Apple Vision Pro per presentazioni al cliente. "
    "Come posso procedere con l'acquisto?"
)})

print("\n📤 RISPOSTA FINALE:", r3["output"])

  SCENARIO 3 — Prodotto non a catalogo


> Entering new AgentExecutor chain...

Invoking: `search_product_catalog` with `{'query': 'Apple Vision Pro'}`


{'status': 'NOT_FOUND', 'message': "Nessun prodotto trovato per 'Apple Vision Pro'. Prodotto NON presente a catalogo."}Ciao Alessandro Corti,

Grazie per averci contattato per l'acquisto di un Apple Vision Pro. Tuttavia, dopo aver verificato il catalogo aziendale, abbiamo trovato che il prodotto non è presente a catalogo.

In base alla nostra policy aziendale, non possiamo inventare prezzi o approvazioni per prodotti non presenti nel catalogo. Pertanto, non possiamo procedere con l'acquisto di un Apple Vision Pro per te.

Se hai bisogno di un dispositivo per le tue presentazioni al cliente, ti consiglio di considerare opzioni alternative disponibili nel catalogo aziendale. Se hai bisogno di ulteriore assistenza o hai domande, non esitare a chiedere.

Cordiali saluti,
[Il tuo Assistente Aziendale per gli Acquisti Hardware]

> Finished 

## 6. Riepilogo dei Risultati

In [8]:
print("=" * 60)
print("  RIEPILOGO TEST MVP — CRITERI DI SUCCESSO")
print("=" * 60)

scenari = [
    ("Scenario 1", "Approvazione automatica",  r1["output"], ["approv", "✅"]),
    ("Scenario 2", "Escalation al manager",    r2["output"], ["escal", "manager", "⚠️"]),
    ("Scenario 3", "Rifiuto (fuori catalogo)", r3["output"], ["rifiut", "catalogo", "❌"]),
]

tutti_ok = True
for nome, atteso, risposta, keywords in scenari:
    trovato = any(k.lower() in risposta.lower() for k in keywords)
    esito   = "✅ OK" if trovato else "⚠️  Verifica manuale"
    if not trovato: tutti_ok = False
    print(f"\n  {nome}: {atteso}")
    print(f"  Verifica automatica: {esito}")

print("\n" + "=" * 60)
print("  🏆 OBIETTIVO RAGGIUNTO!" if tutti_ok else "  ⚠️  Verifica manuale suggerita.")
print("=" * 60)

  RIEPILOGO TEST MVP — CRITERI DI SUCCESSO

  Scenario 1: Approvazione automatica
  Verifica automatica: ✅ OK

  Scenario 2: Escalation al manager
  Verifica automatica: ✅ OK

  Scenario 3: Rifiuto (fuori catalogo)
  Verifica automatica: ✅ OK

  🏆 OBIETTIVO RAGGIUNTO!


## 7. Cosa Fa AgentExecutor Internamente?

Con `verbose=True` hai già visto i log. Vediamo la struttura della risposta completa:

In [9]:
# Esegui senza verbose per ispezionare la struttura dell'output
executor_quiet = AgentExecutor(
    agent=agent, tools=tools, verbose=False,
    return_intermediate_steps=True,   # restituisce anche i passaggi intermedi
)

result_detail = executor_quiet.invoke({"input": (
    "Ciao, sono Mario Rossi (E4521). Voglio il ThinkPad P16s per Project Delta."
)})

print("Chiavi nell'output:", list(result_detail.keys()))
print("\nRisposta finale:\n", result_detail["output"])
print("\nPassaggi intermedi (intermediate_steps):")
for i, (action, obs) in enumerate(result_detail["intermediate_steps"], 1):
    print(f"  Step {i}: tool='{action.tool}'  args={action.tool_input}")
    obs_str = json.dumps(obs, ensure_ascii=False) if isinstance(obs, dict) else str(obs)
    print(f"          result={obs_str[:120]}")

Chiavi nell'output: ['input', 'output', 'intermediate_steps']

Risposta finale:
 Ciao Mario Rossi,

Ho verificato le informazioni e posso confermare che il ThinkPad P16s è presente nel nostro catalogo aziendale.

Il costo di questo prodotto è di 1650€, che rientra nella fascia di spesa standard per il tuo ruolo di Senior Developer nel progetto Project Delta. Inoltre, il budget residuo che hai a disposizione è di 1800€, quindi il costo del prodotto è completamente coperto dal tuo budget.

La richiesta è quindi APPROVAZIONE AUTOMATICA.

Se hai bisogno di ulteriori informazioni o hai domande, non esitare a contattarmi.

Cordiali saluti,
[Il mio nome]

Passaggi intermedi (intermediate_steps):
  Step 1: tool='get_employee_budget'  args={'employee_id': 'E4521'}
          result={"status": "OK", "employee_id": "E4521", "name": "Mario Rossi", "department": "Engineering", "role": "Senior Developer",
  Step 2: tool='search_product_catalog'  args={'query': 'ThinkPad P16s'}
          result={"stat

## 8. 💡 Esercizi Pratici

### Esercizio A — Nuovo Scenario
Scrivi e testa un quarto scenario: **Alessandro Corti (E9900)** chiede un Thunderbolt Dock (190€).
Atteso: approvazione automatica (accessori ≤ 200€ non richiedono approvazione speciale).

In [10]:
# TODO — Esercizio A
richiesta_a = "..."  # scrivi la richiesta di Alessandro Corti

# Decommenta per eseguire:
# r_a = executor.invoke({"input": richiesta_a})
# print(r_a["output"])

print("💡 Scrivi la richiesta e decommenta le righe sopra.")

💡 Scrivi la richiesta e decommenta le righe sopra.


### Esercizio B — Modifica la Descrizione del Tool
Copia `get_employee_budget` e modifica la docstring rimuovendo le istruzioni su *quando* chiamarlo.
Poi esegui il flow e osserva: il modello lo chiama ancora? Cosa cambia nel comportamento?

In [11]:
# ── TOOL 1: get_employee_budget ──────────────────────────────────────────
@tool
def get_employee_budget(employee_id: str) -> dict:
    """Recupera budget residuo, ruolo, dipartimento e progetto attivo di un dipendente.
    Chiamare SEMPRE prima di valutare qualsiasi richiesta per conoscere budget disponibile
    e fascia di spesa autorizzata. Argomento: employee_id (badge ID, es. 'E4521')."""
    employees = json.loads((DATA_DIR / "employees.json").read_text())
    if employee_id not in employees:
        return {"status": "NOT_FOUND", "message": f"Dipendente '{employee_id}' non trovato."}
    emp = employees[employee_id]
    return {
        "status": "OK",
        "employee_id": employee_id,
        "name": emp["name"],
        "department": emp["department"],
        "role": emp["role"],
        "budget_residuo_eur": emp["budget_residuo"],
        "active_project": emp.get("active_project"),
        "manager_email": emp["manager_email"]
    }


# ── TOOL 2: search_product_catalog ───────────────────────────────────────
@tool
def search_product_catalog(query: str) -> dict:
    """Cerca prodotti nel catalogo hardware aziendale per nome o categoria.
    Usare SEMPRE per verificare se il prodotto richiesto esiste a catalogo, il suo prezzo ufficiale e la disponibilità.
    Categorie disponibili: laptop, monitor, accessory.
    Esempi di query: 'ThinkPad', 'laptop', 'monitor', 'Dell'."""
    products = json.loads((DATA_DIR / "products.json").read_text())
    q = query.lower().strip()
    results = {}
    for pid, p in products.items():
        name_l = p["name"].lower()
        cat_l = p["category"].lower()
        roles_l = [r.lower() for r in p["target_roles"]]
        if q in name_l or q in cat_l or any(q == r for r in roles_l):
            results[pid] = p
    if not results:
        terms = [t for t in q.split() if t not in ["il", "la", "un", "una", "del", "per", "di"]]
        for pid, p in products.items():
            name_l = p["name"].lower()
            if sum(1 for t in terms if t in name_l or t in p["category"].lower()) >= 2:
                results[pid] = p
    if not results:
        return {"status": "NOT_FOUND", "message": f"Nessun prodotto trovato per '{query}'. Prodotto NON presente a catalogo."}
    return {"status": "OK", "risultati": results, "totale": len(results)}


# Ispeziona lo schema generato automaticamente
print("📋 Schema generato da @tool (automatico, no JSON manuale):")
for t in [get_employee_budget, search_product_catalog]:
    print(f"\n  Tool: {t.name}")
    print(f"  Descr: {t.description[:80]}...")
    print(f"  Args:  {t.args}")


📋 Schema generato da @tool (automatico, no JSON manuale):

  Tool: get_employee_budget
  Descr: Recupera budget residuo, ruolo, dipartimento e progetto attivo di un dipendente....
  Args:  {'employee_id': {'title': 'Employee Id', 'type': 'string'}}

  Tool: search_product_catalog
  Descr: Cerca prodotti nel catalogo hardware aziendale per nome o categoria.
    Usare S...
  Args:  {'query': {'title': 'Query', 'type': 'string'}}


---
### ✅ Recap Lab 2

- Abbiamo usato `@tool` per definire i tool: **nessuno schema JSON manuale**.
- Abbiamo costruito il prompt con `ChatPromptTemplate` e il placeholder `agent_scratchpad`.
- Abbiamo usato `create_tool_calling_agent` + `AgentExecutor` per il loop ReAct: **nessun `for` manuale**.
- Abbiamo testato i 3 scenari di successo del Giorno 3.
- Abbiamo visto `return_intermediate_steps=True` per ispezionare il ragionamento.

➡️ **Prossimo notebook:** `03_Pattern_Human_Approval.ipynb` — aggiungiamo un checkpoint umano per le decisioni ad alto impatto.